# XGBoost

In [ ]:
import pandas as pd
import numpy as np
import re
import xgboost as xgb
from sklearn.model_selection import GroupKFold, GridSearchCV, cross_validate
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import f1_score, make_scorer, classification_report
from tabulate import tabulate
from pathlib import Path
import warnings
import re

# Nascondo i warning
warnings.filterwarnings('ignore')

# Definisco il percorso dei file
FILE_PATH = Path('/Users/francesco/Tesi/BC-ML4/dataset/cleaned')

# Lista dei csv su cui fare training
datasets = {
    't2_medsam': FILE_PATH / 't2_medsam_masks.csv',
    't2_preprocessed': FILE_PATH / 't2_preprocessed_masks.csv',
    't2_original': FILE_PATH / 't2_original_masks.csv',
    'medsam_dynamic': FILE_PATH / 'medsam_dynamic.csv',
    'preprocessed_dynamic': FILE_PATH / 'preprocessed_dynamic.csv',
    'original_dynamic': FILE_PATH / 'original_dynamic.csv'
}

# Training


In [ ]:
def training(file_path, csv_name):
    # Vado a leggere il csv
    df = pd.read_csv(file_path)

    # Definisco le colonne target
    original_target_list = ['PR [SII]', 'ER [SII]', 'KI67 [%]']

    # Vado a rimuovere le lesioni (righe) non valide
    df_validi = df.dropna(subset=original_target_list).copy()

    # Trasformo tutto in valori binari
    df_validi['PR_class'] = (df_validi['PR [SII]'] > 0.5).astype(int)
    df_validi['ER_class'] = (df_validi['ER [SII]'] > 0.5).astype(int)
    df_validi['KI67_class'] = (df_validi['KI67 [%]'] >= 20).astype(int)

    # Lista finale delle colonne target binarizzate
    final_target_list = ['PR_class', 'ER_class', 'KI67_class']

    # Preparo le feature (X) e i target (y)
    features_to_drop = ['Patient ID', 'lesion idx', 'tumor/benign', 'GRADE', 'isTN', 'Breast'] + original_target_list + final_target_list
    features = df_validi.drop(columns=features_to_drop, errors='ignore')

    target = df_validi[final_target_list]
    groups = df_validi['Patient ID']

    # Riempio a Nan se sono rimasti vuoti
    features = features.fillna(features.mean())
    features.columns = [re.sub(r'\[|\]|<', '', col) for col in features.columns]

    # Imposto la di cross-validation
    cv = GroupKFold(n_splits=5)

    # Definizione Modello Base e MultiOutput
    base_model = xgb.XGBClassifier(tree_method='hist', random_state=42, n_jobs=1)
    multi_output_model = MultiOutputClassifier(base_model)

    iperparametri = {
      # 100 va bene, ma aggiungo 50. Con pochi dati, a volte meno alberi = generalizzazione migliore.
      'estimator__n_estimators': [50, 100],          
      
      # CRITICO: Ho tolto 5. Con 82 pazienti, depth=5 crea foglie con 1-2 persone (memorizzazione pura).
      # Depth 1 (Decision Stump) o 2 sono molto più sicuri.
      'estimator__max_depth': [1, 2],                
      
      # Va bene 0.05. Aggiungo 0.1 perché se usi depth=1 (alberi deboli), potresti dover imparare più in fretta.
      'estimator__learning_rate': [0.05, 0.1],       
      
      # Ho alzato 0.6 a 0.8. Se usi 0.6 su 60 pazienti di train, l'albero ne vede solo 36. Troppo rumore.
      'estimator__subsample': [0.8, 1.0],            
      
      'estimator__colsample_bytree': [0.6, 0.8],     
      
      # CRITICO: Ho tolto 1. "1" permette di isolare un singolo paziente anomalo.
      # "3" o "5" costringono il modello a trovare pattern comuni ad almeno 3-5 pazienti.
      'estimator__min_child_weight': [3, 5],         
      
      # Ho aggiunto 0.5 per essere più conservativo (richiede una riduzione di loss maggiore per fare uno split).
      'estimator__gamma': [0.1, 0.5],                  
      
      'estimator__reg_alpha': [0.1, 1],              
    }


    # Scorer personalizzato per F1 Macro medio sui 3 target
    def multi_f1_scorer(y_true, y_pred):
        # Mi assicuro che gli input siano numpy array
        y_true = np.array(y_true)
        y_pred = np.array(y_pred)
        
        scores = []
        for i in range(y_true.shape[1]):
            s = f1_score(y_true[:, i], y_pred[:, i], average='macro', zero_division=0)
            scores.append(s)
        return np.mean(scores)
    
    scorer = make_scorer(multi_f1_scorer)

    print(f"\nInizio Grid Search per: {csv_name}")
    
    # Configurazione GridSearchCV
    grid_search = GridSearchCV(
        estimator=multi_output_model,
        param_grid=iperparametri,
        cv=cv,
        scoring=scorer,
        n_jobs=-1,  # Parallelizzazione a livello di griglia
        verbose=1,
        return_train_score=False
    )

    # Esecuzione Grid Search
    grid_search.fit(features, target, groups=groups)

    # Prendo i risultati migliori
    best_params = grid_search.best_params_
    best_score = grid_search.best_score_
    best_model = grid_search.best_estimator_

    
    fold_reports = []
    # Uso cross_validate manuale sugli indici per catturare i report testuali
    for train_idx, test_idx in cv.split(features, target, groups):
        X_train, X_test = features.iloc[train_idx], features.iloc[test_idx]
        y_train, y_test = target.iloc[train_idx], target.iloc[test_idx]
        
        # Mi copio il modello migliore per non rovinarlo
        model_clone =  grid_search.best_estimator_ 
        model_clone.fit(X_train, y_train)
        y_pred = model_clone.predict(X_test)
        
        # Genero report per ogni target
        report_dict = {}
        for i, col in enumerate(final_target_list):
            report_dict[col] = classification_report(y_test.iloc[:, i], y_pred[:, i], output_dict=True, zero_division=0)
        fold_reports.append(report_dict)

    
    final_result = [{
        # Pulisco le chiavi rimuovendo 'estimator__' per la stampa
        **{k.replace('estimator__', ''): v for k, v in best_params.items()},
        'mean_score': best_score,
        'std_score': grid_search.cv_results_['std_test_score'][grid_search.best_index_],
        'fold_reports': fold_reports
    }]

    return final_result

# Vado a stampare il risultato in un formato leggibile

In [ ]:
def print_grid_search_results(results_per_dataset):
    """
    Stampa i risultati della Grid Search in modo organizzato e leggibile
    """
    print("\n" + "=" * 80)
    print(" " * 25 + "RIEPILOGO DEI MIGLIORI RISULTATI")
    print("=" * 80)

    summary_data = []

    for name, metrics_list in results_per_dataset.items():
        # Contiene solo n singolo elemento
        best_result = metrics_list[0] 

        print(f"\n{'─' * 80}")
        print(f" Dataset: {name}")
        print(f"{'─' * 80}")
        print(f"\n Performance: F1-score = {best_result['mean_score']:.3f} ± {best_result['std_score']:.3f}\n")

        # Tabella Iperparametri
        print("Iperparametri Ottimali:")
        params_table = [
            ['n_estimators', best_result.get('n_estimators')],
            ['max_depth', best_result.get('max_depth')],
            ['learning_rate', best_result.get('learning_rate')],
            ['subsample', best_result.get('subsample')],
            ['colsample_bytree', best_result.get('colsample_bytree')],
            ['min_child_weight', best_result.get('min_child_weight')],
            ['gamma', best_result.get('gamma')],
            ['reg_alpha (L1)', best_result.get('reg_alpha')],
            ['reg_lambda (L2)', best_result.get('reg_lambda')]
        ]
        print(tabulate(params_table, headers=['Parametro', 'Valore'], tablefmt='simple'))

        print("\n Metriche di Classificazione per Target (Media sui fold):\n")
        target_names = ['PR_class', 'ER_class', 'KI67_class']

        
        first_fold_report = best_result['fold_reports'][0]

        for target_name in target_names:
            current_target_report = first_fold_report[target_name]
            
            rows = []
            # Verifica quali classi sono presenti nel report (a volte manca la classe 0 o 1 se il fold è piccolo)
            classes = [c for c in ['0', '1'] if c in current_target_report]
            
            for cls in classes:
                rows.append([
                    f"Classe {cls}",
                    f"{current_target_report[cls]['precision']:.3f}",
                    f"{current_target_report[cls]['recall']:.3f}",
                    f"{current_target_report[cls]['f1-score']:.3f}",
                    int(current_target_report[cls]['support'])
                ])

            print(f"  {target_name}:")
            print(tabulate(rows, headers=['', 'Precision', 'Recall', 'F1-score', 'Support'],
                         tablefmt='simple', colalign=('left', 'center', 'center', 'center', 'center')))
            print()

        summary_data.append([
            name,
            f"{best_result['mean_score']:.3f}",
            f"{best_result['std_score']:.3f}",
            best_result.get('max_depth'),
            best_result.get('learning_rate'),
            best_result.get('n_estimators')
        ])

    print("\n" + "=" * 80)
    print(" " * 25 + "CONFRONTO TRA TUTTI I DATASET")
    print("=" * 80 + "\n")

    summary_data.sort(key=lambda x: float(x[1]), reverse=True)

    print(tabulate(summary_data,
                   headers=['Dataset', 'F1-score', 'Std Dev', 'Max Depth', 'LR', 'N Est.'],
                   tablefmt='grid',
                   floatfmt=('.3f', '.3f', '.3f', '.2f', 'g', '.1f', 'g')))

# Lettura dei file

In [ ]:
# Esegui il training per tutti i dataset
results_per_dataset = {}
for name, file_path in datasets.items():
    results_per_dataset[name] = training(file_path, name)

# Usa la nuova funzione per stampare i risultati
print_grid_search_results(results_per_dataset)
